# VisDrone YOLOv8 — Colab Runner
Thin launcher. All logic lives in the GitHub repo.
**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
# 1. Mount Drive (data lives here)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone repo
!git clone https://github.com/YOUR_USERNAME/visdrone-yolo.git
%cd visdrone-yolo

In [ ]:
# 3. Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# 4. Unzip data (only needed once — skip if already extracted)
import zipfile, os

DRIVE_ROOT = '/content/drive/MyDrive/visdrone'
EXTRACT_TO = '/content/visdrone_raw'
os.makedirs(EXTRACT_TO, exist_ok=True)

for fname in ['VisDrone2019-DET-train.zip', 'VisDrone2019-DET-val.zip', 'VisDrone2019-DET-test-dev.zip']:
    fpath = f'{DRIVE_ROOT}/{fname}'
    print(f'Unzipping {fname}...')
    with zipfile.ZipFile(fpath, 'r') as z:
        z.extractall(EXTRACT_TO)
print('Done.')

In [ ]:
# 5. Convert annotations to YOLO format
!python data/prepare.py --root /content/visdrone_raw --output data/processed

In [ ]:
# 6. Update dataset yaml path for Colab
import yaml
with open('configs/visdrone.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['path'] = '/content/visdrone-yolo/data/processed'
with open('configs/visdrone.yaml', 'w') as f:
    yaml.dump(cfg, f)
print('yaml updated:', cfg['path'])

In [ ]:
# 7. Train
!python train.py --model yolov8s.pt --epochs 50 --batch 16

In [ ]:
# 8. Evaluate
!python eval.py --weights runs/train/visdrone_yolov8s/weights/best.pt

In [ ]:
# 9. Save best weights back to Drive
import shutil
shutil.copy(
    'runs/train/visdrone_yolov8s/weights/best.pt',
    '/content/drive/MyDrive/visdrone/best_yolov8s.pt'
)
print('Saved to Drive.')